# 3.1.2, Proof of the Cartan relations on a 0-form and a 1-form

**Goal.** Sequentially close all seven core Cartan relations,
step-by-step in the system, first on a 0-form $f$ and then on a 1-form
$\omega$.

| # | Relation | Meaning |
|---|---|---|
| 1 | $\iota_X \iota_Y + \iota_Y \iota_X = 0$ | interior products anti-commute |
| 2 | $(\iota_X d + d \iota_X)\,\omega = \mathcal{L}_X \omega$ | Cartan's magic formula |
| 3 | $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]_{VF}}$ | Lie / interior commutator |
| 4 | $[d, \mathcal{L}_X] = 0$ | $d$ and $\mathcal{L}_X$ commute |
| 5 | $d^2 = 0$ | nilpotency of the exterior derivative |
| 6 | $[\mathcal{L}_X, \mathcal{L}_Y] = \mathcal{L}_{[X,Y]_{VF}}$ | commutator of Lie derivatives |

For each relation we present two sub-headings: first the **0-form**
$f$, then the **1-form** $\omega$. The proof chains are rendered to
LaTeX via `display_chain`.

## Strategy

We set up two engines, each carrying the rules natural to its
form-degree.

| Form | Engine | Why |
|---|---|---|
| $f$ (0-form) | `default_engine` + 3 closure axioms | $\iota_X(f) = 0$, $\mathcal{L}_X(f) = X(f)$, $\iota_X(df) = X(f)$, Cartan magic, $d^2 = 0$, $\iota_X^2 = 0$, all built-in. The closure axioms supply $\mathcal{L}^2$ and $[X,Y]$ closure. |
| $\omega$ (1-form) | `intrinsic_engine_with_closure` + three 0-form helpers | Opens every relation in the form $\text{multi\_eval}(\cdot, V_1, \dots, V_p)$ via Koszul/Cartan; once interiors fall to a 0-form, three extra rules ($\iota_X(0\text{-form}) = 0$, $\iota_X^2 = 0$, $\mathcal{L}_X(0\text{-form}) = X(\cdot)$) zero them out. |

The proof goes through one call: `prove_intrinsic_equivalence(LHS,
RHS, engine=…, registry=…)`. It runs the engine to a fixed point on
`Sum(LHS, -RHS)`; if the result reduces to zero we get a `ProofChain`,
otherwise a `ProofFailure`.

For relation **(4)**, $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]_{VF}}$,
on a 1-form both sides reduce to a 0-form scalar; we explicitly write
those scalars in the system's $\text{multi\_eval}(\omega, Y) \equiv
\omega(Y)$ form. The intrinsic $\iota$/$\mathcal{L}$ rules fire on
this shape; using a *flow*-mode $\mathcal{L}_X$ also opens the
reduction $\mathcal{L}_X(\omega(Y)) \to X(\omega(Y))$ on the
0-form scalar.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Setup, symbols, engines, helper function

Symbols used below:

- $f$: a 0-form (function, `Graded(degree=0)`).
- $\omega$: a 1-form (`Graded(degree=1)`).
- $X, Y, Z$: degree-0 vector fields (`Derivation`).

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.algebra.lie_bracket_vf import lie_bracket_vf
from jacopy.calculus.closure_axioms import (
    LieBracketVfAntiSymmetryDefinition,
    LieBracketVfJacobiDefinition,
    VfActCommutatorDefinition,
)
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.interior import interior
from jacopy.calculus.intrinsic_engine import (
    intrinsic_engine_with_closure,
    prove_intrinsic_equivalence,
)
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Integer, Neg, Sum, Symbol
from jacopy.core.multi_eval import multi_eval
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.proof.expansion import (
    ExpansionEngine,
    IotaOnZeroFormDefinition,
    IotaSquaredZeroDefinition,
    LieDerivativeOnZeroFormDefinition,
    default_engine,
)

reg = PropertyRegistry()
f = Symbol("f");     reg.declare(f, Graded(degree=0))
omega = Symbol("ω"); reg.declare(omega, Graded(degree=1))
X = Derivation("X", 0)
Y = Derivation("Y", 0)
Z = Derivation("Z", 0)

print("f =", f, "  degree =", reg.get(f, Graded).degree)
print("ω =", omega, "  degree =", reg.get(omega, Graded).degree)

f = f   degree = 0
ω = ω   degree = 1


### 1.1. Two engines

**0-form engine.** `default_engine` already carries everything needed
specifically on 0-forms: Cartan magic ($\mathcal{L}_X = d \circ
\iota_X + \iota_X \circ d$), $d^2 = 0$, $\iota_X^2 = 0$, $\iota_X(f) = 0$,
$\mathcal{L}_X(f) = X(f)$, $\iota_X(df) = X(f)$, plus operator
distribution. Three closure axioms are added: $X(Y(f)) - Y(X(f)) =
[X,Y]_{VF}(f)$, $[X,Y]_{VF} + [Y,X]_{VF} = 0$, and VF-Jacobi, these
are the only extra layer needed for relation (7).

**1-form engine.** `intrinsic_engine_with_closure` carries three
intrinsic rules ($\iota$, $\mathcal{L}$, $d$) + four multi-eval
helpers + three closure axioms + the `IotaActAsScalar` bridge. Three
*0-form collapse* rules are added on top: for relations (1)/(2),
zeroing $\iota_X$ when $\iota_Y(\omega)$ has fallen to a scalar
0-form, and the operator form $\iota_X^2 = 0$.

In [3]:
# 0-form engine: default + closure axioms
defs_zf = list(default_engine(registry=reg, d_squared_mode="axiom").definitions)
defs_zf += [
    VfActCommutatorDefinition(),
    LieBracketVfAntiSymmetryDefinition(),
    LieBracketVfJacobiDefinition(),
]
engine_zf = ExpansionEngine(defs_zf)

# 1-form engine: intrinsic + closure + 0-form collapse rules
defs_of = list(intrinsic_engine_with_closure().definitions)
defs_of += [
    IotaOnZeroFormDefinition(registry=reg),
    IotaSquaredZeroDefinition(),
    LieDerivativeOnZeroFormDefinition(registry=reg),
]
engine_of = ExpansionEngine(defs_of)

print(f"0-form engine: {len(engine_zf.definitions)} rules")
print(f"1-form engine: {len(engine_of.definitions)} rules")


0-form engine: 11 rules
1-form engine: 15 rules


### 1.2. Helper function

A single call interface: `prove(label, lhs, rhs, engine)`. It prints
the step count and returns the `ProofChain`; later cells render it as
LaTeX via `display_chain(chain)`.

In [4]:
def prove(label, lhs, rhs, engine):
    chain = prove_intrinsic_equivalence(lhs, rhs, engine=engine, registry=reg)
    print(f"{label} -> closed in {len(chain)} steps")
    return chain


## 2. Relation 1: iota anti-commute

$$
\iota_X \iota_Y + \iota_Y \iota_X = 0.
$$

### 2.1. On a 0-form $f$

Since $\iota_X(f) = 0$, each term is separately zero, so the sum is
of course zero.

In [5]:
lhs = Sum(Act(interior(X), Act(interior(Y), f)),
           Act(interior(Y), Act(interior(X), f)))
chain = prove("(0-form) ι_X ι_Y f + ι_Y ι_X f = 0", lhs, Integer(0), engine_zf)
display_chain(chain)


(0-form) ι_X ι_Y f + ι_Y ι_X f = 0 -> closed in 5 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\iota_Y\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_X\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_X\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_Y\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\left(0 + 0\right) - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [6]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ι_X(f) = 0 on 0-forms [axiom]
    ι_Y(f)
 -> 0

[2] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(0)
 -> 0

[3] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(f)
 -> 0

[4] ι_X(f) = 0 on 0-forms [axiom]
    ι_Y(0)
 -> 0

[5] simplify 
    ((0 + 0) + (-0))
 -> 0



### 2.2. Evaluation on a 1-form

Since $\omega$ is a 1-form, $\iota_Y(\omega)$ and
$\iota_X(\omega)$ are scalar 0-forms; the inner engine zeros out a
$\iota_X$ that has dropped to a 0-form via its rule.

In [7]:
lhs = Sum(Act(interior(X), Act(interior(Y), omega)),
           Act(interior(Y), Act(interior(X), omega)))
chain = prove("(1-form) ι_X ι_Y ω + ι_Y ι_X ω = 0", lhs, Integer(0), engine_of)
display_chain(chain)


(1-form) ι_X ι_Y ω + ι_Y ι_X ω = 0 -> closed in 3 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\iota_X\!\left(\iota_Y\!\left(\omega\right)\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_Y\!\left(\iota_X\!\left(\omega\right)\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\left(0 + 0\right) - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [8]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(ι_Y(ω))
 -> 0

[2] ι_X(f) = 0 on 0-forms [axiom]
    ι_Y(ι_X(ω))
 -> 0

[3] simplify 
    ((0 + 0) + (-0))
 -> 0



## 3. Relation 3: Cartan's magic formula

$$
(\iota_X \, d + d \, \iota_X)\,\omega \;=\; \mathcal{L}_X \omega.
$$

### 3.1. On a 0-form $f$

$f$ is a 0-form: $\iota_X(df) = X(f)$ and $d(\iota_X f) = d(0) = 0$,
sum is $X(f) = \mathcal{L}_X f$.

In [9]:
lhs = Sum(Act(interior(X), Act(default_d, f)),
           Act(default_d, Act(interior(X), f)))
rhs = Act(lie_derivative(X), f)
chain = prove("(0-form) (ι_X d + d ι_X) f = L_X f", lhs, rhs, engine_zf)
display_chain(chain)


(0-form) (ι_X d + d ι_X) f = L_X f -> closed in 9 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\iota_X\!\left(d\!\left(f\right)\right) \to X\!\left(f\right) \quad \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)} \\
\iota_X\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
L_X\!\left(f\right) \to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\left(X\!\left(f\right) + d\!\left(0\right)\right) - \left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) \to \left(X\!\left(f\right) + 0\right) - \left(d\!\left(\iota_X\!\left(f\right)\right) + \iota_X\!\left(d\!\left(f\right)\right)\right) \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(X\!\left(f\right) + 0\right) - \left(d\!\left(\iota_X\!\left(f\right)\right) + \iota_X\!\left(d\!\left(f\right)\right)\right) \to X\!\left(f\right) - d\!\left(\iota_X\!\left(f\right)\right) - \iota_X\!\left(d\!\left(f\right)\right) \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
\iota_X\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_X\!\left(d\!\left(f\right)\right) \to X\!\left(f\right) \quad \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)} \\
X\!\left(f\right) - d\!\left(0\right) - X\!\left(f\right) \to X\!\left(f\right) - 0 - X\!\left(f\right) \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
X\!\left(f\right) - 0 - X\!\left(f\right) \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [10]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ι_X(df) = X(f) [axiom]
    ι_X(d(f))
 -> X(f)

[2] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(f)
 -> 0

[3] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(f)
 -> ((d * ι_X)(f) + (ι_X * d)(f))

[4] product-rule 
    ((X(f) + d(0)) + (-((d * ι_X)(f) + (ι_X * d)(f))))
 -> ((X(f) + 0) + (-(d(ι_X(f)) + ι_X(d(f)))))

[5] simplify 
    ((X(f) + 0) + (-(d(ι_X(f)) + ι_X(d(f)))))
 -> (X(f) + (-d(ι_X(f))) + (-ι_X(d(f))))

[6] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(f)
 -> 0

[7] ι_X(df) = X(f) [axiom]
    ι_X(d(f))
 -> X(f)

[8] product-rule 
    (X(f) + (-d(0)) + (-X(f)))
 -> (X(f) + (-0) + (-X(f)))

[9] simplify 
    (X(f) + (-0) + (-X(f)))
 -> 0



### 3.2. On a 1-form $\omega$, evaluated against $Y$

For a 1-form $\omega$, both sides are 1-forms; we evaluate against a
common vector field $Y$ and run the Koszul expansion. The intrinsic
rules cancel the bracket term and equate the two sides syntactically.

In [11]:
lhs = Sum(multi_eval(Act(interior(X), Act(default_d, omega)), Y),
           multi_eval(Act(default_d, Act(interior(X), omega)), Y))
rhs = multi_eval(Act(lie_derivative(X), omega), Y)
chain = prove("(1-form, eval Y) ((ι_X d + d ι_X) ω)(Y) = (L_X ω)(Y)",
              lhs, rhs, engine_of)
display_chain(chain)


(1-form, eval Y) ((ι_X d + d ι_X) ω)(Y) = (L_X ω)(Y) -> closed in 6 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\iota_X\!\left(d\!\left(\omega\right)\right)\right)\!\left(Y\right) \to \left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right) \quad \text{[\ensuremath{\iota}\_X intrinsic: (\ensuremath{\iota}\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = \ensuremath{\omega}(X, Y\_1, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right) \to X\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Y]_{VF}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\iota_X\!\left(\omega\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_X\!\left(\omega\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_X\!\left(\omega\right)\right) \to Y\!\left(\omega\!\left(X\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Y\right) \to X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(\left(X\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Y]_{VF}\right)\right) + Y\!\left(\omega\!\left(X\right)\right)\right) - \left(X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right)\right) \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [12]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ι_X intrinsic: (ι_X ω)(Y_1, …) = ω(X, Y_1, …) [axiom]
    ι_X(d(ω))(Y)
 -> d(ω)(X, Y)

[2] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ω)(X, Y)
 -> (X(ω(Y)) + (-Y(ω(X))) + (-ω([X,Y]_VF)))

[3] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_X(ω))(Y)
 -> Y(ι_X(ω))

[4] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    Y(ι_X(ω))
 -> Y(ω(X))

[5] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)(Y)
 -> (X(ω(Y)) + (-ω([X,Y]_VF)))

[6] simplify 
    (((X(ω(Y)) + (-Y(ω(X))) + (-ω([X,Y]_VF))) + Y(ω(X))) + (-(X(ω(Y)) + (-ω([X,Y]_VF)))))
 -> 0



## 4. Relation 4: $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]_{VF}}$

$$
\mathcal{L}_X \iota_Y - \iota_Y \mathcal{L}_X \;=\; \iota_{[X,Y]_{VF}}.
$$

### 4.1. On a 0-form $f$

On $f$ each term is 0; the equality is trivial, the engine finds it
via Cartan magic + the $\iota$-on-0-form route.

In [13]:
lhs = Sum(Act(lie_derivative(X), Act(interior(Y), f)),
           Neg(Act(interior(Y), Act(lie_derivative(X), f))))
rhs = Act(interior(lie_bracket_vf(X, Y)), f)
chain = prove("(0-form) [L_X, ι_Y] f = ι_[X,Y] f", lhs, rhs, engine_zf)
display_chain(chain)


(0-form) [L_X, ι_Y] f = ι_[X,Y] f -> closed in 12 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\iota_Y\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
L_X\!\left(0\right) \to \left(d \, \iota_X\right)\!\left(0\right) + \left(\iota_X \, d\right)\!\left(0\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_X\!\left(f\right) \to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\iota_{[X,Y]_{VF}}\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\left(\left(\left(d \, \iota_X\right)\!\left(0\right) + \left(\iota_X \, d\right)\!\left(0\right)\right) - \iota_Y\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right)\right) - 0 \to \left(\left(0 + 0\right) - \left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + \iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(\left(0 + 0\right) - \left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + \iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 \to -\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) - \iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right) \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
\iota_X\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_Y\!\left(d\!\left(0\right)\right) \to Y\!\left(0\right) \quad \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)} \\
\iota_X\!\left(d\!\left(f\right)\right) \to X\!\left(f\right) \quad \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)} \\
\iota_Y\!\left(X\!\left(f\right)\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
-Y\!\left(0\right) - 0 \to -0 - 0 \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
-0 - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [14]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ι_X(f) = 0 on 0-forms [axiom]
    ι_Y(f)
 -> 0

[2] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(0)
 -> ((d * ι_X)(0) + (ι_X * d)(0))

[3] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(f)
 -> ((d * ι_X)(f) + (ι_X * d)(f))

[4] ι_X(f) = 0 on 0-forms [axiom]
    ι_[X,Y]_VF(f)
 -> 0

[5] product-rule 
    ((((d * ι_X)(0) + (ι_X * d)(0)) + (-ι_Y(((d * ι_X)(f) + (ι_X * d)(f))))) + (-0))
 -> (((0 + 0) + (-(ι_Y(d(ι_X(f))) + ι_Y(ι_X(d(f)))))) + (-0))

[6] simplify 
    (((0 + 0) + (-(ι_Y(d(ι_X(f))) + ι_Y(ι_X(d(f)))))) + (-0))
 -> ((-ι_Y(d(ι_X(f)))) + (-ι_Y(ι_X(d(f)))))

[7] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(f)
 -> 0

[8] ι_X(df) = X(f) [axiom]
    ι_Y(d(0))
 -> Y(0)

[9] ι_X(df) = X(f) [axiom]
    ι_X(d(f))
 -> X(f)

[10] ι_X(f) = 0 on 0-forms [axiom]
    ι_Y(X(f))
 -> 0

[11] product-rule 
    ((-Y(0)) + (-0))
 -> ((-0) + (-0))

[12] simplify 
    ((-0) + (-0))
 -> 0



### 4.2. On a 1-form $\omega$, evaluated against $Y$

Both sides reduce to a 0-form scalar; we write the expressions
explicitly in $\text{multi\_eval}(\omega, Y) \equiv \omega(Y)$
form so the intrinsic rules fire. Since the left-hand
$\mathcal{L}_X(\iota_Y \omega) = \mathcal{L}_X(\omega(Y))$ has
dropped to a 0-form scalar, we use the *flow*-mode $\mathcal{L}_X$,
this opens the step $\mathcal{L}_X(\omega(Y)) \to X(\omega(Y))$.

In [15]:
from jacopy.calculus.lie_derivative import LieDerivative

L_X_flow = LieDerivative(X, definition="flow")  # to drop to a 0-form scalar
L_X_full = lie_derivative(X)                    # cartan-mode for the form side

lhs = Sum(
    Act(L_X_flow, multi_eval(omega, Y)),                  # L_X(ω(Y))
    Neg(multi_eval(Act(L_X_full, omega), Y)),             # (L_X ω)(Y)
)
rhs = multi_eval(omega, lie_bracket_vf(X, Y))              # ω([X,Y])
chain = prove("(1-form, eval Y) L_X(ω(Y)) − (L_X ω)(Y) = ω([X,Y])",
              lhs, rhs, engine_of)
display_chain(chain)


(1-form, eval Y) L_X(ω(Y)) − (L_X ω)(Y) = ω([X,Y]) -> closed in 3 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
L_X\!\left(\omega\!\left(Y\right)\right) \to X\!\left(\omega\!\left(Y\right)\right) \quad \text{[L\_X(f) = X(f) on 0-forms (flow)]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Y\right) \to X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(X\!\left(\omega\!\left(Y\right)\right) - \left(X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right)\right)\right) - \omega\!\left([X,Y]_{VF}\right) \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [16]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X(f) = X(f) on 0-forms (flow) [axiom]
    L_X(ω(Y))
 -> X(ω(Y))

[2] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)(Y)
 -> (X(ω(Y)) + (-ω([X,Y]_VF)))

[3] simplify 
    ((X(ω(Y)) + (-(X(ω(Y)) + (-ω([X,Y]_VF))))) + (-ω([X,Y]_VF)))
 -> 0



## 5. Relation 5: $[d, \mathcal{L}_X] = 0$

$$
d \, \mathcal{L}_X \;=\; \mathcal{L}_X \, d.
$$

### 5.1. On a 0-form $f$

For $f$, $\mathcal{L}_X f = X(f)$ and $df$ is a 1-form; the *flow*
axiom $\mathcal{L}_X \circ d = d \circ \mathcal{L}_X$ fires
directly.

In [17]:
lhs = Sum(Act(default_d, Act(lie_derivative(X), f)),
           Neg(Act(lie_derivative(X), Act(default_d, f))))
chain = prove("(0-form) (d L_X − L_X d) f = 0", lhs, Integer(0), engine_zf)
display_chain(chain)


(0-form) (d L_X − L_X d) f = 0 -> closed in 9 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
L_X\!\left(f\right) \to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_X\!\left(d\!\left(f\right)\right) \to \left(d \, \iota_X\right)\!\left(d\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(d\!\left(f\right)\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\left(d\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) - \left(\left(d \, \iota_X\right)\!\left(d\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(d\!\left(f\right)\right)\right)\right) - 0 \to \left(\left(d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right) - \left(d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right) + \iota_X\!\left(d\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(\left(d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right) - \left(d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right) + \iota_X\!\left(d\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 \to d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) - \iota_X\!\left(d\!\left(d\!\left(f\right)\right)\right) \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
\iota_X\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
d\!\left(d\!\left(0\right)\right) \to 0 \quad \text{[d² = 0]\,(axiom)} \\
d\!\left(d\!\left(f\right)\right) \to 0 \quad \text{[d² = 0]\,(axiom)} \\
\iota_X\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
0 - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [18]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(f)
 -> ((d * ι_X)(f) + (ι_X * d)(f))

[2] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(d(f))
 -> ((d * ι_X)(d(f)) + (ι_X * d)(d(f)))

[3] product-rule 
    ((d(((d * ι_X)(f) + (ι_X * d)(f))) + (-((d * ι_X)(d(f)) + (ι_X * d)(d(f))))) + (-0))
 -> (((d(d(ι_X(f))) + d(ι_X(d(f)))) + (-(d(ι_X(d(f))) + ι_X(d(d(f)))))) + (-0))

[4] simplify 
    (((d(d(ι_X(f))) + d(ι_X(d(f)))) + (-(d(ι_X(d(f))) + ι_X(d(d(f)))))) + (-0))
 -> (d(d(ι_X(f))) + (-ι_X(d(d(f)))))

[5] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(f)
 -> 0

[6] d² = 0 [axiom]
    d(d(0))
 -> 0

[7] d² = 0 [axiom]
    d(d(f))
 -> 0

[8] ι_X(f) = 0 on 0-forms [axiom]
    ι_X(0)
 -> 0

[9] simplify 
    (0 + (-0))
 -> 0



### 5.2. On a 1-form $\omega$, evaluated against $(Y, Z)$

For a 1-form $\omega$ the difference is a 2-form; we evaluate against
$(Y, Z)$ and zero it via Koszul expansion + commutator + Jacobi
steps.

In [19]:
lhs = Sum(multi_eval(Act(default_d, Act(lie_derivative(X), omega)), Y, Z),
           Neg(multi_eval(Act(lie_derivative(X), Act(default_d, omega)), Y, Z)))
chain = prove("(1-form, eval Y,Z) ((d L_X − L_X d) ω)(Y, Z) = 0",
              lhs, Integer(0), engine_of)
display_chain(chain)


(1-form, eval Y,Z) ((d L_X − L_X d) ω)(Y, Z) = 0 -> closed in 15 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(d\!\left(L_X\!\left(\omega\right)\right)\right)\!\left(Y,\, Z\right) \to Y\!\left(\left(L_X\!\left(\omega\right)\right)\!\left(Z\right)\right) - Z\!\left(\left(L_X\!\left(\omega\right)\right)\!\left(Y\right)\right) - \left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Z\right) \to X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Y\right) \to X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) \to X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]_{VF]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_X\!\left(d\!\left(\omega\right)\right)\right)\!\left(Y,\, Z\right) \to X\!\left(\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right)\right) - \left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) - \left(d\!\left(\omega\right)\right)\!\left(Y,\, [X,Z]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right) \to Y\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([Y,Z]_{VF}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) \to [X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left([X,Y]_{VF}\right)\right) - \omega\!\left([[X,Y]_{VF,Z]_{VF}}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left(Y,\, [X,Z]_{VF}\right) \to Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - [X,Z]_{VF}\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([Y,[X,Z]_{VF]_{VF}}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(\left(Y\!\left(X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right)\rig

In [20]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(L_X(ω))(Y, Z)
 -> (Y(L_X(ω)(Z)) + (-Z(L_X(ω)(Y))) + (-L_X(ω)([Y,Z]_VF)))

[2] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)(Z)
 -> (X(ω(Z)) + (-ω([X,Z]_VF)))

[3] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)(Y)
 -> (X(ω(Y)) + (-ω([X,Y]_VF)))

[4] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)([Y,Z]_VF)
 -> (X(ω([Y,Z]_VF)) + (-ω([X,[Y,Z]_VF]_VF)))

[5] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(d(ω))(Y, Z)
 -> (X(d(ω)(Y, Z)) + (-d(ω)([X,Y]_VF, Z)) + (-d(ω)(Y, [X,Z]_VF)))

[6] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ω)(Y, Z)
 -> (Y(ω(Z)) + (-Z(ω(Y))) + (-ω([Y,Z]_VF)))

[7] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_

## 6. Relation 6: $d^2 = 0$

$$
d \circ d \;=\; 0.
$$

### 6.1. On a 0-form $f$

A single axiom: $d(df) = 0$.

In [21]:
lhs = Act(default_d, Act(default_d, f))
chain = prove("(0-form) d(df) = 0", lhs, Integer(0), engine_zf)
display_chain(chain)


(0-form) d(df) = 0 -> closed in 2 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
d\!\left(d\!\left(f\right)\right) \to 0 \quad \text{[d² = 0]\,(axiom)} \\
0 - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [22]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] d² = 0 [axiom]
    d(d(f))
 -> 0

[2] simplify 
    (0 + (-0))
 -> 0



### 6.2. On a 1-form $\omega$, evaluated against $(X, Y, Z)$

For a 1-form $\omega$, $d(d\omega)$ is a 3-form; we evaluate against
$(X, Y, Z)$. The expansion closes via two Koszul layers + the
commutator $X(Y(g)) - Y(X(g)) = [X,Y]_{VF}(g)$ on every bracket pair
+ a three-bracket Jacobi chain.

In [23]:
lhs = multi_eval(Act(default_d, Act(default_d, omega)), X, Y, Z)
chain = prove("(1-form, eval X,Y,Z) (d(dω))(X, Y, Z) = 0",
              lhs, Integer(0), engine_of)
display_chain(chain)


(1-form, eval X,Y,Z) (d(dω))(X, Y, Z) = 0 -> closed in 15 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(d\!\left(d\!\left(\omega\right)\right)\right)\!\left(X,\, Y,\, Z\right) \to X\!\left(\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right)\right) - Y\!\left(\left(d\!\left(\omega\right)\right)\!\left(X,\, Z\right)\right) + Z\!\left(\left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right)\right) - \left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) + \left(d\!\left(\omega\right)\right)\!\left([X,Z]_{VF},\, Y\right) - \left(d\!\left(\omega\right)\right)\!\left([Y,Z]_{VF},\, X\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right) \to Y\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([Y,Z]_{VF}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left(X,\, Z\right) \to X\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Z]_{VF}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right) \to X\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Y]_{VF}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) \to [X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left([X,Y]_{VF}\right)\right) - \omega\!\left([[X,Y]_{VF,Z]_{VF}}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left([X,Z]_{VF},\, Y\right) \to [X,Z]_{VF}\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([[X,Z]_{VF,Y]_{VF}}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\omega\right)\right)\!\left([Y,Z]_{VF},\, X\right) \to [Y,Z]_{VF}\!\left(\omega\!\left(X\right)\right) - X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([[Y,Z]_{VF,X]_{VF}}\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(X\!\left(Y\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([Y,Z]_{VF}\

In [24]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(d(ω))(X, Y, Z)
 -> (X(d(ω)(Y, Z)) + (-Y(d(ω)(X, Z))) + Z(d(ω)(X, Y)) + (-d(ω)([X,Y]_VF, Z)) + d(ω)([X,Z]_VF, Y) + (-d(ω)([Y,Z]_VF, X)))

[2] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ω)(Y, Z)
 -> (Y(ω(Z)) + (-Z(ω(Y))) + (-ω([Y,Z]_VF)))

[3] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ω)(X, Z)
 -> (X(ω(Z)) + (-Z(ω(X))) + (-ω([X,Z]_VF)))

[4] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ω)(X, Y)
 -> (X(ω(Y)) + (-Y(ω(X))) + (-ω([X,Y]_VF)))

[5] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ω)([X,Y]_VF, Z)
 -> ([X,Y]_VF(ω(Z)) + (-Z(ω([X,Y]_VF))) + (-ω([[X,Y]_VF,Z]_VF)))

[6] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [ax

## 7. Relation 7: $[\mathcal{L}_X, \mathcal{L}_Y] = \mathcal{L}_{[X,Y]_{VF}}$

$$
\mathcal{L}_X \mathcal{L}_Y - \mathcal{L}_Y \mathcal{L}_X
   \;=\; \mathcal{L}_{[X,Y]_{VF}}.
$$

### 7.1. On a 0-form $f$

On $f$ both sides are scalar: $X(Y(f)) - Y(X(f)) = [X,Y]_{VF}(f)$,
exactly what `VfActCommutatorDefinition` does.

In [25]:
lhs = Sum(Act(lie_derivative(X), Act(lie_derivative(Y), f)),
           Neg(Act(lie_derivative(Y), Act(lie_derivative(X), f))))
rhs = Act(lie_derivative(lie_bracket_vf(X, Y)), f)
chain = prove("(0-form) [L_X, L_Y] f = L_[X,Y] f", lhs, rhs, engine_zf)
display_chain(chain)


(0-form) [L_X, L_Y] f = L_[X,Y] f -> closed in 30 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
L_Y\!\left(f\right) \to \left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_X\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) \to \left(d \, \iota_X\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_X\!\left(f\right) \to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_Y\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) \to \left(d \, \iota_Y\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) + \left(\iota_Y \, d\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_{[X,Y]_{VF}}\!\left(f\right) \to \left(d \, \iota_{[X,Y]_{VF}}\right)\!\left(f\right) + \left(\iota_{[X,Y]_{VF}} \, d\right)\!\left(f\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\left(\left(\left(d \, \iota_X\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right)\right) - \left(\left(d \, \iota_Y\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) + \left(\iota_Y \, d\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right)\right)\right) - \left(\left(d \, \iota_{[X,Y]_{VF}}\right)\!\left(f\right) + \left(\iota_{[X,Y]_{VF}} \, d\right)\!\left(f\right)\right) \to \left(\left(\left(d\!\left(\iota_X\!\left(d\!\left(\iota_Y\!\left(f\right)\right)\right)\right) + d\!\left(\iota_X\!\left(\iota_Y\!\left(d\!\left(f\right)\right)\right)\right)\right) + \left(\iota_X\!\left(d\!\left(d\!\left(\iota_Y\!\left(f\right)\right)\right)\right) + \iota_X\!\left(d\!\left(\iota_Y\!\left(d\!\left(f\right)\right)\right)\right)\right)\right) - \left(\left(d\!\left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right)\right) + d\!\left(\iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) + \left(\iota_Y\!\left(d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right)\right) + \iota_Y\!\left(d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right)\right)\right) - \left(d\!\left(\iota_{[X,Y]_{VF}}\!\left(f\right)\right) + \iota_{[X,Y]_{VF}}\!\left(d\!\left(f\right)\right)\right) \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(\left(\left(d\!\left(\iota_X\!\left(d\!\left(\iota_Y\!\left(f\right)\right)\right)\right) + d\!\left(\iota_X\!\left(\iota_Y\!\left(d\!\left(f\right)\right)\right)\right)\right) + \left(\iota_X\!\left(d\!\left(d\!\left(\iota_Y\!\left(f\right)\right)\right)\right) + \iota_X\!\left(d\!\left(\iota_Y\!\left(d\!\left(f\right)\right)\right)\right)\right)\right) - \left(\left(d\!\left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right)\right) + d\!\left(\iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) + \left(\iota_Y\!\left(d\!\left(d\

In [26]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_Y(f)
 -> ((d * ι_Y)(f) + (ι_Y * d)(f))

[2] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(((d * ι_Y)(f) + (ι_Y * d)(f)))
 -> ((d * ι_X)(((d * ι_Y)(f) + (ι_Y * d)(f))) + (ι_X * d)(((d * ι_Y)(f) + (ι_Y * d)(f))))

[3] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X(f)
 -> ((d * ι_X)(f) + (ι_X * d)(f))

[4] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_Y(((d * ι_X)(f) + (ι_X * d)(f)))
 -> ((d * ι_Y)(((d * ι_X)(f) + (ι_X * d)(f))) + (ι_Y * d)(((d * ι_X)(f) + (ι_X * d)(f))))

[5] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_[X,Y]_VF(f)
 -> ((d * ι_[X,Y]_VF)(f) + (ι_[X,Y]_VF * d)(f))

[6] product-rule 
    ((((d * ι_X)(((d * ι_Y)(f) + (ι_Y * d)(f))) + (ι_X * d)(((d * ι_Y)(f) + (ι_Y * d)(f)))) + (-((d * ι_Y)(((d * ι_X)(f) + (ι_X * d)(f))) + (ι_Y * d)(((d * ι_X)(f) + (ι_X * d)(f)))))) + (-((d * ι_[X,Y]_VF)(f) + (ι_[X,Y]_VF * d)(f))))
 -> ((((d(ι_X(d(ι_Y(f)))) + d(ι_X(ι_Y(d(f))))) + (ι_X(d(d(ι_Y(f)

### 7.2. On a 1-form $\omega$, evaluated against $Z$

For a 1-form $\omega$ the difference is a 1-form; evaluate against
$Z$ and close in a single pipeline via intrinsic expansion +
commutator/Jacobi chain.

In [27]:
lhs = Sum(multi_eval(Act(lie_derivative(X), Act(lie_derivative(Y), omega)), Z),
           Neg(multi_eval(Act(lie_derivative(Y), Act(lie_derivative(X), omega)), Z)))
rhs = multi_eval(Act(lie_derivative(lie_bracket_vf(X, Y)), omega), Z)
chain = prove("(1-form, eval Z) ([L_X, L_Y] ω)(Z) = (L_[X,Y] ω)(Z)",
              lhs, rhs, engine_of)
display_chain(chain)


(1-form, eval Z) ([L_X, L_Y] ω)(Z) = (L_[X,Y] ω)(Z) -> closed in 12 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(L_X\!\left(L_Y\!\left(\omega\right)\right)\right)\!\left(Z\right) \to X\!\left(\left(L_Y\!\left(\omega\right)\right)\!\left(Z\right)\right) - \left(L_Y\!\left(\omega\right)\right)\!\left([X,Z]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_Y\!\left(\omega\right)\right)\!\left(Z\right) \to Y\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([Y,Z]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_Y\!\left(\omega\right)\right)\!\left([X,Z]_{VF}\right) \to Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([Y,[X,Z]_{VF]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_Y\!\left(L_X\!\left(\omega\right)\right)\right)\!\left(Z\right) \to Y\!\left(\left(L_X\!\left(\omega\right)\right)\!\left(Z\right)\right) - \left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Z\right) \to X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) \to X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]_{VF]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_{[X,Y]_{VF}}\!\left(\omega\right)\right)\!\left(Z\right) \to [X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([[X,Y]_{VF,Z]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(\left(X\!\left(Y\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([Y,Z]_{VF}\right)\right) - \left(Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([Y,[X,Z]_{VF]_{VF}}\right)\right)\right) - \left(Y\!\left(X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right)\right) - \left(X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]_{VF]_{VF}}\right)\right)\right)\right) - \left([X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([[X,Y]_{VF,Z]_{VF}}\right)\right) \to \left(\left(\left(X\!\left(Y\!\left(\omega\!\left(Z\right)\right)\right) - X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right)\right) - \left(Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([Y,[X,Z]_{VF]_{VF}}\right)\right)\right) - \left(\left(Y\!\left(X\!\left(\omega\!\left(Z\right)\right)\right) - Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right)\right) - \left(X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]_{V

In [28]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(L_Y(ω))(Z)
 -> (X(L_Y(ω)(Z)) + (-L_Y(ω)([X,Z]_VF)))

[2] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_Y(ω)(Z)
 -> (Y(ω(Z)) + (-ω([Y,Z]_VF)))

[3] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_Y(ω)([X,Z]_VF)
 -> (Y(ω([X,Z]_VF)) + (-ω([Y,[X,Z]_VF]_VF)))

[4] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_Y(L_X(ω))(Z)
 -> (Y(L_X(ω)(Z)) + (-L_X(ω)([Y,Z]_VF)))

[5] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)(Z)
 -> (X(ω(Z)) + (-ω([X,Z]_VF)))

[6] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_X(ω)([Y,Z]_VF)
 -> (X(ω([Y,Z]_VF)) + (-ω([X,[Y,Z]_VF]_VF)))

[7] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_[X,Y]_VF(ω)(Z)
 -> ([X,Y]_VF(ω(Z)) + (-ω([[X,Y]_VF,Z]_VF)))

[8] 

## Conclusion

All seven Cartan relations close syntactically, first on a 0-form
$f$, then on a 1-form $\omega$, through a single
`prove_intrinsic_equivalence` call combined with the appropriate
engine.

| # | Relation | 0-form | 1-form |
|---|---|---|---|
| 1 | $\iota \iota$ anti-commute | trivial via $\iota_X(f) = 0$ | $\iota_X(\iota_Y\omega) = 0$ via 0-form collapse |
| 2 | Cartan magic | $\iota_X(df) + d(\iota_X f) = X(f)$ | Koszul expansion + bracket cancellation |
| 3 | $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]}$ | both sides 0 | $\iota$'s written via `multi_eval(ω, Y)`; *flow* $\mathcal{L}_X$ drops to a scalar |
| 4 | $[d, \mathcal{L}_X] = 0$ | flow axiom | two-layer Koszul + Jacobi |
| 5 | $d^2 = 0$ | axiom $d(df) = 0$ | 3-form evaluation + commutator/Jacobi |
| 6 | $[\mathcal{L}_X, \mathcal{L}_Y] = \mathcal{L}_{[X,Y]}$ | `VfActCommutator` | intrinsic expansion + commutator |

All proofs are readable as LaTeX via `display_chain`; the
`Definition` from which each step originated is recorded in
`ProofStep.rule`.